# Persian Poetry Generation in the Style of Ferdowsi with GPT-2

This notebook fine-tunes `HooshvareLab/gpt2-fa`, a Persian GPT-2 model from Hugging Face, on a corpus of Ferdowsi's poetry (`ferdousi.txt`) to produce the second hemistich of a couplet from the first. The corpus holds one hemistich per line, which gives 49,608 pairs of a first and a second hemistich, each ending with a `<sep>` marker; the pairs are split 90/10 into training and test sets. The model is trained to predict, at each position of the tokenized first hemistich, the token at the same position of the second hemistich, with cross-entropy loss and early stopping on a validation loss. The notebook shows generations before fine-tuning and, after fine-tuning, beam-search continuations of 20 training and 32 test hemistichs.

## Setup

Import the libraries and load the `HooshvareLab/gpt2-fa` tokenizer with an added `[PAD]` padding token. The next cells select the GPU if one is available and show that the tokenizer encodes `<sep>` as the single token 9.

In [1]:
%%capture
from sklearn.model_selection import train_test_split
from transformers import GPT2Tokenizer, AutoModelForCausalLM ,GPT2Model
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from transformers import AutoModelForCausalLM, AutoTokenizer, AdamW
from torchvision import transforms
import torch.nn.functional as F
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import files
import os
import numpy as np
model_name = "HooshvareLab/gpt2-fa"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
print(tokenizer.encode("<sep>"))

[9]


## Data

`ferdousi.txt` starts with a two-line header and then holds one hemistich per line. Even-numbered lines go to `A` (first hemistichs) and odd-numbered lines to `B` (second hemistichs), each with `<sep>` appended, and dropping the first pair removes the header. Both lists are tokenized and padded to their longest sequence, split 90/10 with `random_state=42`, and served in shuffled batches of 32.

In [4]:
A = []
B = []
with open('ferdousi.txt', 'r') as file:
    for line_num, line in enumerate(file, start=0):
        # Even line numbers hold first hemistichs
        if line_num % 2 != 1:
            A.append(line.strip()+'<sep>')
        # Odd line numbers hold second hemistichs
        else:
            B.append(line.strip()+'<sep>')

# Drop the first pair, which is the file's two-line header
A ,B= A[1:], B[1:]


print(B[0])
print(A[0])
print('num of verses:',len(A))
print('num of verses:',len(B))

کزین برتر اندیشه برنگذرد<sep>
به نام خداوند جان و خرد<sep>
num of verses: 49608
num of verses: 49608


In [5]:
input_tokenized = tokenizer(A, return_tensors='pt', padding=True, truncation=False)
target_tokenized = tokenizer(B, return_tensors='pt', padding=True, truncation=False)

In [6]:
data_input = input_tokenized['input_ids']
data_target =target_tokenized['input_ids']
attention_input = input_tokenized['attention_mask']
attention_target = target_tokenized['attention_mask']

input_train, input_test, target_train, target_test, input_attention_train , input_attention_test    = train_test_split(data_input,
                                                                      data_target, attention_input,test_size=0.1, random_state=42)

In [7]:
vers1 = tokenizer.decode(input_train[0], skip_special_tokens=True)
vers2 = tokenizer.decode(target_train[0], skip_special_tokens=True)
print(vers1)
print(vers2)

وگر در میان دو رویه سپاه <sep>
بگردی بلاف از پی نام و جاه <sep>


In [8]:
class CustumeDataset(Dataset):
    def __init__(self, input_ids, attention_mask_input, target_ids):
        self.input_ids = input_ids
        self.target_ids = target_ids
        self.attention_mask_input = attention_mask_input

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_mask_input[idx], self.target_ids[idx]

train_set = CustumeDataset(input_train, input_attention_train, target_train)
test_set = CustumeDataset(input_test, input_attention_test, target_test)

trainloader = DataLoader(train_set, batch_size=32, shuffle=True,num_workers =2)
testloader = DataLoader(test_set, batch_size=32, shuffle=True,num_workers =2)

## Model

Load `HooshvareLab/gpt2-fa` with its language-modeling head and resize its token embeddings for the added padding token. The next two cells generate a continuation of up to 26 tokens for one training hemistich before fine-tuning, as a baseline.

In [21]:
%%capture
config = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=config)
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

In [10]:
for data,attention,target in trainloader:
  data = data.to(device)
  attention = attention.to(device)
  target = target.to(device)
  break

In [11]:
output = model.generate(data,max_length=26)
generated_text = tokenizer.decode(output[10], skip_special_tokens=True)
input_text = tokenizer.decode(data[10], skip_special_tokens=True)
print((input_text))
print(generated_text[len(input_text):])


ز من باد بر شاه ایران درود <sep>
  است که در آن از دو واژهٔ «م» و «


## Fine-Tuning

`val_loop` computes the loss on the test loader but returns inside its loop, so it uses only the first batch, which changes on every call because the loader is shuffled. The training loop fine-tunes the model with AdamW (learning rate 0.001) for up to 50 epochs. The loss is the cross-entropy between the logits at each position of the first hemistich and the token at the same position of the second hemistich, padding positions included.

Every 100 iterations, the loop records the validation loss and a moving average of the last 10 values. When the moving average decreases, the current state is kept as `checkpoint`; after three consecutive increases, that state is loaded back and training stops. `checkpoint` holds `state_dict()` references rather than copies, so loading it back does not restore earlier weights. The printed validation loss is divided by the number of test batches, although it comes from a single batch.

In [12]:
def val_loop(testloader,model,loss_function):
  model.eval()
  with torch.no_grad():
    for i, (inputs,attention, targets) in enumerate(testloader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        attention = attention.to(device)

        outputs = model(inputs,attention_mask=attention, labels=targets)
        logits = outputs.logits

        # Returns the loss of the first batch only
        return loss_function(logits.view(-1, logits.size(-1)), targets.view(-1))


In [22]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters() , lr= 0.001 , eps= 1e-8 )
min_val_loss = float('inf')
epochs = 50
n = 0
itter = 0
val_loss = []
mean_val_losses = []
# Training loop
for epoch in range(epochs):
    running_loss = 0
    model.train()

    for (inputs,attention, targets) in tqdm(trainloader):
        itter += 1
        inputs = inputs.to(device)
        targets = targets.to(device)
        attention = attention.to(device)

        optimizer.zero_grad()

        outputs = model(inputs,attention_mask=attention, labels=targets)
        logits = outputs.logits

        # Cross-entropy between the logits at each input position and the target token at the same position
        loss = loss_function(logits.view(-1, logits.size(-1)), targets.view(-1))

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Every 100 iterations: validation loss and its moving average over the last 10 values
        if itter % 100 == 0:
            val_loss.append(val_loop(testloader,model,loss_function).item())
            # The first value is repeated 10 times so that the moving average is defined
            if len(val_loss)<10:
              val_loss = 10*val_loss

            mean_val_loss = np.mean(val_loss[-10:])
            mean_val_losses.append(mean_val_loss)

            if len(mean_val_losses) > 1:
                if mean_val_losses[-2] > mean_val_losses[-1]:
                  n = 0
                  # Keep the current state (state_dict() references, not copies)
                  checkpoint = {'model_state_dict': model.state_dict(),
                      'optimizer_state_dict': optimizer.state_dict()}

                elif mean_val_losses[-2] <= mean_val_losses[-1]:
                    n+=1
                    print(f"Validation loss is increasing:{val_loss[-1]}")
                if n == 3 :
                  model.load_state_dict(checkpoint['model_state_dict'])
                  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                  break
    if n == 3:
      break
    print(f'Epoch [{epoch+1}/{epochs}], Training Loss: {running_loss/len(trainloader):.4f}, Validation Loss: {val_loss[-1]/len(testloader):.4f}')
torch.save(checkpoint, "checkpoint.pth")


100%|██████████| 1396/1396 [03:37<00:00,  6.41it/s]


Epoch [1/50], Training Loss: 3.7427, Validation Loss: 0.0237


  0%|          | 5/1396 [00:01<05:14,  4.42it/s]

Validation loss is increasing:3.664842128753662


 58%|█████▊    | 805/1396 [02:05<01:49,  5.41it/s]

...

Validation loss is increasing:3.647052526473999


 15%|█▍        | 209/1396 [00:33<03:58,  4.97it/s]

Validation loss is increasing:3.6896321773529053


 22%|██▏       | 307/1396 [00:48<02:53,  6.27it/s]

Validation loss is increasing:3.7954695224761963


## Generation

Generate continuations of up to 26 tokens for one batch of training hemistichs and one batch of test hemistichs with beam search (10 beams) that blocks repeated bigrams. `temperature`, `top_k`, and `top_p` are also passed, but they have no effect because sampling is not enabled.

In [23]:
for i, (inputs,attention, targets) in enumerate(trainloader):
   inputs = inputs.to(device)
   targets = targets.to(device)
   attention = attention.to(device)
   break

In [24]:
print('Generating for training data')
generated_text = model.generate(inputs,max_length=26,
        temperature=0.7,num_beams=10,
        no_repeat_ngram_size=2,
        top_k=50,top_p=0.95,
        pad_token_id=tokenizer.pad_token_id)
for j in range(20):
        input_text = tokenizer.decode(inputs[j], skip_special_tokens=True)
        output_text = tokenizer.decode(generated_text[j], skip_special_tokens=True)
        print(f"Input{j}:     {input_text.replace('<sep>','')}")
        print(f"Generated{j}: {output_text.replace('<sep>','')[0:]}")

Generating for training data
Input0:     همم گنج و بوم است و هم چارپای 
Generated0: همم گنج و بوم است و هم چارپای     گاه   و   به و و به 
Input1:     بریزم ز تن خون انباردار 
Generated1: بریزم ز تن خون انباردار   خوار   گار و   کارزار   خوار
Input2:     نهادند سوی فرامرز روی 
Generated2: نهادند سوی فرامرز روی    و و    گاه می نه و و
Input3:     ازان تازی اسپان کش آمد گزین 
Generated3: ازان تازی اسپان کش آمد گزین   چین    چین   کین کین   زمین زمین 
Input4:     پیاده بیامد به نزدیک شاه 
Generated4: پیاده بیامد به نزدیک شاه     گاه گاه   راه راه و   سپاه 
Input5:     به تیزی بیامد به نزدیک شاه 
Generated5: به تیزی بیامد به نزدیک شاه   گاه    کلاه راه   گاه   راه به
Input6:     خدای جهان را نباشد نیاز 
Generated6: خدای جهان را نباشد نیاز    نیاز نیاز و گاه باز   باز نیاز
Input7:     کسی را ندانم که روز نبرد 
Generated7: کسی را ندانم که روز نبرد   گرد    گرد  ژ و   کرد نبرد کرد
Input8:     بر رستم آمد همانگاه گیو 
Generated8: بر رستم آمد همانگاه گیو   نیو   و و   نیو   گی گی
Input9:     به

In [25]:
for i, (inputs,attention, targets) in enumerate(testloader):
   inputs = inputs.to(device)
   targets = targets.to(device)
   attention = attention.to(device)
   break

In [26]:
print('Generating for test data')
generated_text = model.generate(inputs,max_length=26,
        temperature=0.7,num_beams=10,
        no_repeat_ngram_size=2,
        top_k=20,top_p=0.95,
        pad_token_id=tokenizer.pad_token_id)
for j in range(generated_text.shape[0]):
        input_text = tokenizer.decode(inputs[j], skip_special_tokens=True)
        output_text = tokenizer.decode(generated_text[j], skip_special_tokens=True)
        print(f"Input{j}:     {input_text.replace('<sep>','')}")
        print(f"Generated{j}: {output_text.replace('<sep>','')[0:]}")

Generating for test data
Input0:     نباید که در پیش خسرو شود 
Generated0: نباید که در پیش خسرو شود   واندواندواند  شود شودواند نماند  
Input1:     همان به که او را برپهلوان 
Generated1: همان به که او را برپهلوان   و    و   دل به بر   بک
Input2:     و گر شاه و فرزانگان این به جای 
Generated2: و گر شاه و فرزانگان این به جای     پایمای   و و   رهن  مای
Input3:     درفش سرافراز خاقان و تاج 
Generated3: درفش سرافراز خاقان و تاج    عاج گاه عاج و   تاج تاج تاج عاج
Input4:     بیامد چو نزدیک ایشان رسید 
Generated4: بیامد چو نزدیک ایشان رسید    دید دید   کشید   بدید   برکشید
Input5:     چنان بد که روزی به نخچیرگاه 
Generated5: چنان بد که روزی به نخچیرگاه     گاه شاه راه سپاه و   و
Input6:     چو گشتاسپ را دید بر تخت عاج 
Generated6: چو گشتاسپ را دید بر تخت عاج   تاج    عاج عاج و تاج   تاج تاج عاج
Input7:     بیاراست پیلان و برخاست غو 
Generated7: بیاراست پیلان و برخاست غو   و    تو نو   نو راست   و
Input8:     ز طایر یکی دختش آمد چو ماه 
Generated8: ز طایر یکی دختش آمد چو ماه   و    آمد بودش ک